# Esercizio 1

Caricate il musk dataset ed eliminate le prime colonne non numeriche


In [2]:
import pandas as pd

# --- 1. CARICAMENTO DATI ---

labels=['molecule', 'conformation']+list(range(166)) +['class']
df = pd.read_csv('http://www.lacascia.it/bd2/clean2.data', names=labels)

Effettuate una classificazione k-NN su tutto il dataset con split fra train e test 75/25


In [3]:
# --- 2. PULIZIA DEI DATI ---

#Bisogna escludere le prime due colonne e l'ultima
data = df.iloc[:, 2:-1].to_numpy(copy=True)
target = df.iloc[:, -1].to_numpy(copy=True)
print(target.shape, data.shape)

(6598,) (6598, 166)


In [ ]:
# 2. Split Train/Test (75/25)
# test_size=0.25 significa che il 25% va nel test set
# random_state=42 serve a rendere l'esperimento riproducibile (mischia sempre allo stesso modo)

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report


# Suddivido il dataset in dati test e dati di allenamento
data_train, data_test, target_train, target_test = train_test_split(data, target, test_size=0.25, random_state=42)

print(f"Dati di training (Studio): {data_train.shape[0]} righe")
print(f"Dati di test (Esame): {data_test.shape[0]} righe")

# 3. Inizializzazione del classificatore K-NN
# Scegliamo k=5 (guarda i 5 vicini più prossimi)

k=5
knn = KNeighborsClassifier(n_neighbors=k)

# 4. Addestramento (fit)
print(f"Addestramento del modello K-NN con k={k} in corso...")
knn.fit(data_train, target_train)

# 5. Predizione
print(f"Predizione sul Test Set in corso...")
target_pred = knn.predict(data_test)

# 6. Valutazione
accuracy = accuracy_score(target_test, target_pred)
print(f"\n--- RISULTATI ---")
print(f"Accuratezza: {accuracy:.2%}")
print("\nReport dettagliato:")
print(classification_report(target_test, target_pred))

Dati di training (Studio): 4948 righe
Dati di test (Esame): 1650 righe
Addestramento del modello K-NN con k=5 in corso...
Predizione sul Test Set in corso...

--- RISULTATI ---
Accuratezza: 96.18%

Report dettagliato:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98      1387
         1.0       0.94      0.81      0.87       263

    accuracy                           0.96      1650
   macro avg       0.95      0.90      0.92      1650
weighted avg       0.96      0.96      0.96      1650



Fate riduzione di dimensionalità  con PCA e classificate nuovamente con k-NN


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. STANDARDIZZAZIONE (passaggio fondamentale per la PCA)

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# 2. APPLICAZIONE DELLA PCA
pca = PCA(n_components=0.95)
data_pca = pca.fit_transform(data_scaled)

#Vediamo quanto abbiamo ridotto
print(f"Dimesioni originali: {data.shape[1]} features")
print(f"Dimesioni dopo PCA (95% info): {data_pca.shape[1]} features")
print(f"Abbiamo eliminato {data.shape[1] - data_pca.shape[1]} colonne inutili")

# 3. Split train test sui dati ridimensionati

X_train_pca, X_test_pca, y_train, y_test = train_test_split(data_pca, target, test_size=0.25, random_state=42)

# 4. Classificazione KNN
k=5
knn_pca = KNeighborsClassifier(n_neighbors=k)

print(f"Addestramento sui dati PCA ({data_pca.shape[1]} dim)...")
knn_pca.fit(X_train_pca, y_train)

print("Predizione in corso...")
y_pred_pca = knn_pca.predict(X_test_pca)

# 5. Valutazione
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"\n--- RISULTATI CON PCA ---")
print(f"Accuratezza Originale (166 dim): 96.18%") # Il tuo risultato precedente
print(f"Accuratezza PCA ({data_pca.shape[1]} dim): {acc_pca:.2%}")
print("\nReport Dettagliato PCA:")
print(classification_report(y_test, y_pred_pca))

Dimesioni originali: 166 features
Dimesioni dopo PCA (95% info): 39 features
Abbiamo eliminato 127 colonne inutili
Addestramento sui dati PCA (39 dim)...
Predizione in corso...

--- RISULTATI CON PCA ---
Accuratezza Originale (166 dim): 96.18%
Accuratezza PCA (39 dim): 96.12%

Report Dettagliato PCA:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      1387
         1.0       0.93      0.81      0.87       263

    accuracy                           0.96      1650
   macro avg       0.95      0.90      0.92      1650
weighted avg       0.96      0.96      0.96      1650



Sul dataset a dimensionalitÃ  ridotta fate un campionamento senza replacement del 40% ed effettuate la classificazione con k-NN sul dataset campionato (split sempre 75/25)


In [10]:
#Con la PCA abbiamo ridotto le colonne, con il campionamento andremo a ridurre le righe

#Recuperiamo il dataframe originale completo 

# 1. CAMPIONAMENTO DEL 40% (Senza rimpiazzo)
# frac=0.4 -> Prendi il 40%
# replace=False -> Una volta estratta una riga, non puoi ripescarla (niente duplicati)
# random_state=42 -> Per avere sempre lo stesso campione (riproducibilità)

# CREIAMO UN DATAFRAME DAI DATI PCA
# data_pca è una matrice di numeri. La trasformiamo in DataFrame.
df_pca = pd.DataFrame(data_pca)
df_pca['target'] = target

print(f"DataFrame PCA creato: {df_pca.shape}")


df_sampled = df_pca.sample(frac=0.4, replace=False, random_state=42)

print(f"Righe originali: {df.shape[0]}")
print(f"Righe dopo il campionamento: {df_sampled.shape[0]}")


# Separiamo di nuovo le righe e le colonne
# X = Tutte le colonne tranne l'ultima ('target')
X_final = df_sampled.iloc[:, :-1].values
# y = Solo la colonna 'target'
y_final = df_sampled['target'].values

# 4. SPLIT TRAINT/TEST

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_final, y_final, random_state=42, test_size=0.25)

# 5. Classificazione K-NN
knn_final = KNeighborsClassifier(n_neighbors=k)

print("Addestramento k-NN sul dataset Ridotto e Campionato...")
knn_final.fit(X_train_f, y_train_f)

print("Predizione...")

y_pred_f = knn_final.predict(X_test_f)

acc_final = accuracy_score(y_test_f, y_pred_f)

print(f"\n--- RISULTATI FINALI (PCA + SAMPLING PANDAS) ---")
print(f"Accuratezza: {acc_final:.2%}")
print("\nReport dettagliato:")
print(classification_report(y_test_f, y_pred_f))


DataFrame PCA creato: (6598, 40)
Righe originali: 6598
Righe dopo il campionamento: 2639
Addestramento k-NN sul dataset Ridotto e Campionato...
Predizione...

--- RISULTATI FINALI (PCA + SAMPLING PANDAS) ---
Accuratezza: 95.91%

Report dettagliato:
              precision    recall  f1-score   support

         0.0       0.97      0.98      0.98       545
         1.0       0.91      0.85      0.88       115

    accuracy                           0.96       660
   macro avg       0.94      0.92      0.93       660
weighted avg       0.96      0.96      0.96       660



# Esercizio 2 - email spam detector
Importate in un DataFrame il dataset che contiene 9997 email etichettate (1=spam, 0=non_spam)

In [1]:
import pandas as pd

labels=['class', 'text']
df = pd.read_csv('http://www.lacascia.it/bd2/combined_data_small.csv', names=labels)

In [2]:
print(df.shape)
df.iloc[:,0].value_counts()

(9997, 2)


1    5321
0    4676
Name: class, dtype: int64

Determinate il dizionario delle parole che compaiono almeno 500 volte in tutto il dataset (nota: dovrebbero essere 650)

Costruite una matrice NumPy 9997x650 che contiene per ciascuna riga la frequenza delle 650 parole piÃ¹ comuni nella  mail (ogni riga corrisponde a una mail del dataset)

Utilizzando la prima colonna del DataFrame come *target* effettuate una classificazione con k-NN e valutatene l'accuratezza

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier



Effettuate una riduzione di dimensionalitÃ  con PCA ed effettuate nuovamente la classificazione.

In [ ]:
from sklearn.decomposition import PCA


# Esercizio 3
Caricate il musk dataset, eliminate le prime colonne non numeriche e riducete la dimensionalitÃ  con PCA tenendo il 99% della varianza.

Effettuate la classificazione con k-NN utilizzando la distanza di Mahalanobis e confrontate l'accuracy con quella ottenuta con la distanza di default (Euclidea).

(Suggerimento: vedete dalla documentazione come si puÃ² utilizzare il KNeighborsClassifier di sklearn con metriche diverse dalla Euclidea o implementate voi l'algoritmo k-NN)

# Esercizio 4
Implementate la shared nearest-neighbor similarity (scegliete voi il numero di vicini da considerare)
ed effettuate nuovamente la classificazione k-NN.